# Limpieza del dataset Titanic

Cargamos train.csv, lo exploramos y lo dejamos limpio, listo para el análisis.

## Cargar librerías y datos

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/train.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Explorar el dataset

### Número de registros

In [3]:
df.shape[0]

891

### Número de columnas

In [4]:
df.shape[1]

12

### Nombre de las variables

In [5]:
df.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='str')

### Tipos de datos

In [6]:
df.dtypes

PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object

### Valores faltantes

In [7]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

### Registros duplicados

In [8]:
df.duplicated().sum()

np.int64(0)

### Estadísticas descriptivas

In [9]:
df.describe(include="all")

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
count,891.000000,891.000000,891.000000,891,891,714.000000,891.000000,891.000000,891,891.000000,204,889
unique,NaN,NaN,NaN,891,2,NaN,NaN,NaN,681,NaN,147,3
top,NaN,NaN,NaN,"Braund, Mr. Owen Harris",male,NaN,NaN,NaN,347082,NaN,G6,S
freq,NaN,NaN,NaN,1,577,NaN,NaN,NaN,7,NaN,4,644
mean,446.000000,0.383838,2.308642,NaN,NaN,29.699118,0.523008,0.381594,NaN,32.204208,NaN,NaN
std,257.353842,0.486592,0.836071,NaN,NaN,14.526497,1.102743,0.806057,NaN,49.693429,NaN,NaN
min,1.000000,0.000000,1.000000,NaN,NaN,0.420000,0.000000,0.000000,NaN,0.000000,NaN,NaN
25%,223.500000,0.000000,2.000000,NaN,NaN,20.125000,0.000000,0.000000,NaN,7.910400,NaN,NaN
50%,446.000000,0.000000,3.000000,NaN,NaN,28.000000,0.000000,0.000000,NaN,14.454200,NaN,NaN
75%,668.500000,1.000000,3.000000,NaN,NaN,38.000000,1.000000,0.000000,NaN,31.000000,NaN,NaN


## Tratar los valores faltantes

`Age` tiene 177 vacíos, así que los llenamos con la mediana (es más estable que el promedio porque no le afectan tanto los valores extremos).

`Cabin` tiene 687 vacíos, o sea más del 75% del total, así que no tiene caso rellenarla. En vez de eso creamos una columna `CabinKnown` que dice si se sabía la cabina del pasajero o no, y quitamos la columna original.

`Embarked` solo tiene 2 vacíos, así que los llenamos con el puerto más común.

In [10]:
df["Age"] = df["Age"].fillna(df["Age"].median())

In [11]:
df["CabinKnown"] = df["Cabin"].notnull().astype(int)
df = df.drop(columns=["Cabin"])

In [12]:
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

## Quitar columnas que no sirven

`PassengerId` es solo un número de fila, y `Name` y `Ticket` son texto libre que no aporta directo al análisis.

In [13]:
df = df.drop(columns=["PassengerId", "Name", "Ticket"])
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,CabinKnown
0,0,3,male,22.0,1,0,7.2500,S,0
1,1,1,female,38.0,1,0,71.2833,C,1
2,1,3,female,26.0,0,0,7.9250,S,0
3,1,1,female,35.0,1,0,53.1000,S,1
4,0,3,male,35.0,0,0,8.0500,S,0


## Crear variables nuevas

`FamilySize` es el total de familiares a bordo (contando al pasajero).

`AgeGroup` agrupa la edad en categorías: Niño (0-12), Joven (13-25), Adulto (26-60) y Adulto mayor (61+).

In [14]:
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

In [15]:
bins = [0, 12, 25, 60, 100]
etiquetas = ["Niño", "Joven", "Adulto", "Adulto mayor"]
df["AgeGroup"] = pd.cut(df["Age"], bins=bins, labels=etiquetas, include_lowest=True)

## Revisar que quedó limpio

In [16]:
df.isnull().sum()

Survived      0
Pclass        0
Sex           0
Age           0
SibSp         0
Parch         0
Fare          0
Embarked      0
CabinKnown    0
FamilySize    0
AgeGroup      0
dtype: int64

In [17]:
df.duplicated().sum()

np.int64(111)

In [18]:
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,CabinKnown,FamilySize,AgeGroup
0,0,3,male,22.0,1,0,7.2500,S,0,2,Joven
1,1,1,female,38.0,1,0,71.2833,C,1,2,Adulto
2,1,3,female,26.0,0,0,7.9250,S,0,1,Adulto
3,1,1,female,35.0,1,0,53.1000,S,1,2,Adulto
4,0,3,male,35.0,0,0,8.0500,S,0,1,Adulto


## Guardar el dataset limpio

In [19]:
df.to_csv("../outputs/resultados/titanic_clean.csv", index=False)